# Baseline lengkap untuk Section 3.3 — SIVPNotebook mandiri. Tidak memerlukan berkas `.py` tambahan.**Isi**1. Konfigurasi dan impor2. Definisi fungsi (sel 3--8, jalankan sekali, tidak menghasilkan keluaran)3. Muat data4. Baseline fitur: colour moments, histogram RGB, ECDF-7 vs ECDF-95. Perbandingan representasi dan uji t berpasangan6. Protokol B: cross-validation bebas kebocoran7. Grid search baseline clustering8. ResNet-50 (opsional, paling lama)9. Ekspor tabel ke LaTeXJalankan sel berurutan dari atas. Sel 1--7 selesai sekitar 5--7 menit.

## 1. Konfigurasi

In [ ]:
# Sesuaikan path berikut. Di Windows pakai garis miring biasa, bukan backslash.EXCEL      = "C:/Users/Ahsan/Downloads/data/data_ekstraksi_rgb_4kelas.xlsx"OUTDIR     = "hasil_baseline"# Folder INDUK citra asli, yang berisi subfolder pink_rose, red_rose,# white_rose, yellow_rose. Hanya dipakai untuk ResNet-50 di sel 8.IMAGE_ROOT = "C:/Users/Ahsan/Downloads/data"HIST_BINS = 4     # histogram RGB gabungan: 4^3 = 64 dimensiN_SEEDS   = 100   # Protokol A

In [ ]:
import ast, json, os, time, warningsimport numpy as npimport pandas as pdfrom scipy import statsfrom scipy.optimize import linear_sum_assignmentfrom sklearn.cluster import KMeans, DBSCAN, AgglomerativeClusteringfrom sklearn.mixture import GaussianMixturefrom sklearn.metrics import (accuracy_score, adjusted_rand_score,                             calinski_harabasz_score, davies_bouldin_score,                             normalized_mutual_info_score,                             precision_recall_fscore_support, silhouette_score)from sklearn.model_selection import StratifiedKFoldfrom sklearn.preprocessing import (MinMaxScaler, QuantileTransformer,                                   StandardScaler)warnings.filterwarnings("ignore")pd.set_option("display.width", 220)pd.set_option("display.max_columns", 60)N_SEEDS_CV       = 20   # Protokol BN_SEEDS_INTERNAL = 20   # silhouette bersifat O(n^2), cukup 20 seedLEVELS = (100, 150, 200)COLS7  = [("R", 100), ("R", 150), ("R", 200),          ("G", 150), ("G", 200), ("B", 150), ("B", 200)]os.makedirs(OUTDIR, exist_ok=True)print("siap")

## 2. Definisi fungsiSel 3 sampai 8 hanya mendefinisikan fungsi dan tidak mencetak apa pun.Jalankan semuanya sekali.

### Pemuatan data

In [ ]:
def load_pixels(excel_path):    """Kembalikan array citra (N, 32, 32, 3), label, dan nama berkas.    Kolom R_values/G_values/B_values disimpan sebagai literal list Python oleh    notebook rose_4_warna.ipynb. Jika berkas Anda menyimpannya sebagai repr    numpy terpotong (mengandung '...'), data mentahnya hilang dan berkas harus    diekstrak ulang dengan str(array.tolist()).    """    df = pd.read_excel(excel_path)    sample = str(df['R_values'].iloc[0])    if '...' in sample:        raise ValueError(            "Kolom piksel tersimpan sebagai repr numpy terpotong. "            "Ekstrak ulang dengan str(row['R_values'].tolist()).")    def parse(s):        return np.asarray(ast.literal_eval(s), dtype=np.uint8)    R = np.stack(df['R_values'].map(parse).values)    G = np.stack(df['G_values'].map(parse).values)    B = np.stack(df['B_values'].map(parse).values)    p = int(df['p'].iloc[0])    imgs = np.stack([R, G, B], axis=-1).reshape(-1, p, p, 3)    return imgs, df['label'].values, df['filename'].values

### Ekstraktor fitur

In [ ]:
def feat_ecdf7(imgs):    """Deskriptor yang diusulkan: ECDF per kanal pada 3 level, 7 koordinat."""    flat = imgs.reshape(len(imgs), -1, 3).astype(np.int16)    ch = {'R': flat[:, :, 0], 'G': flat[:, :, 1], 'B': flat[:, :, 2]}    return np.column_stack([(ch[c] <= t).mean(axis=1) for c, t in COLS7])def feat_ecdf9(imgs):    """Grid penuh 3 kanal x 3 level, 9 koordinat."""    flat = imgs.reshape(len(imgs), -1, 3).astype(np.int16)    ch = {'R': flat[:, :, 0], 'G': flat[:, :, 1], 'B': flat[:, :, 2]}    return np.column_stack([(ch[c] <= t).mean(axis=1)                            for c in ('R', 'G', 'B') for t in LEVELS])def feat_colour_moments(imgs):    """Colour moments Stricker & Orengo: mean, sd, skewness per kanal (d=9)."""    flat = imgs.reshape(len(imgs), -1, 3).astype(np.float64)    mean = flat.mean(axis=1)    sd = flat.std(axis=1)    skew = stats.skew(flat, axis=1)    return np.column_stack([mean, sd, skew])def feat_rgb_histogram(imgs, bins=4):    """Histogram RGB gabungan terkuantisasi, dinormalisasi. d = bins^3.    bins=4 memberi d=64 (dipakai di naskah). bins=8 memberi d=512; laporkan    keduanya jika reviewer menanyakan sensitivitas terhadap kuantisasi.    """    flat = imgs.reshape(len(imgs), -1, 3).astype(np.int32)    q = flat * bins // 256                       # indeks bin per kanal    idx = q[:, :, 0] * bins * bins + q[:, :, 1] * bins + q[:, :, 2]    d = bins ** 3    out = np.zeros((len(imgs), d), dtype=np.float64)    for i in range(len(imgs)):        out[i] = np.bincount(idx[i], minlength=d)    return out / flat.shape[1]

### ResNet-50Hanya dipakai di sel 8.

In [ ]:
def feat_resnet50(imgs=None, image_root=None, filenames=None, batch=64):    """Fitur lapisan penultimate ResNet-50 pra-latih ImageNet (d=2048).    Dua mode:      image_root diberikan -> baca citra ASLI dari disk (direkomendasikan)      image_root None      -> pakai citra 32x32 hasil rekonstruksi, di-upsample    """    import torch    from torch import nn    from torchvision import models, transforms    from PIL import Image    device = 'cuda' if torch.cuda.is_available() else 'cpu'    net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)    net.fc = nn.Identity()                       # ambil keluaran 2048-d    net.eval().to(device)    norm = transforms.Normalize(mean=[0.485, 0.456, 0.406],                                std=[0.229, 0.224, 0.225])    tf_disk = transforms.Compose([transforms.Resize(256),                                  transforms.CenterCrop(224),                                  transforms.ToTensor(), norm])    feats = []    with torch.no_grad():        if image_root is not None:            # Indeks nama berkas -> path, dibangun SEKALI. Versi sebelumnya            # memanggil os.walk untuk setiap berkas, yang berperilaku O(n^2).            index = {}            collisions = set()            for root, _, files in os.walk(image_root):                for f in files:                    if not f.lower().endswith(('.jpg', '.jpeg', '.png')):                        continue                    if f in index:                        collisions.add(f)                    index[f] = os.path.join(root, f)            if collisions:                raise ValueError(                    f'{len(collisions)} nama berkas muncul di lebih dari satu '                    f'subfolder, misalnya {sorted(collisions)[:5]}. Pemetaan '                    'citra ke label menjadi ambigu. Beri prefiks kelas pada '                    'nama berkas, atau ubah fungsi ini agar memakai '                    'subfolder sebagai kunci.')            missing = [f for f in filenames if f not in index]            if missing:                raise FileNotFoundError(                    f'{len(missing)} berkas tidak ditemukan di {image_root}, '                    f'misalnya {missing[:5]}')            paths = [index[f] for f in filenames]            print(f'{len(paths)} citra terindeks dari {image_root}')            for i in range(0, len(paths), batch):                imgs_t = torch.stack([tf_disk(Image.open(p).convert('RGB'))                                      for p in paths[i:i + batch]]).to(device)                feats.append(net(imgs_t).cpu().numpy())                if (i // batch) % 10 == 0:                    print(f'  {min(i + batch, len(paths))}/{len(paths)}',                          end='\r')        else:            x = torch.from_numpy(imgs).permute(0, 3, 1, 2).float() / 255.0            for i in range(0, len(x), batch):                b = torch.nn.functional.interpolate(                    x[i:i + batch], size=224, mode='bilinear',                    align_corners=False)                b = torch.stack([norm(t) for t in b]).to(device)                feats.append(net(b).cpu().numpy())    return np.vstack(feats)

### Transformasi representasi

In [ ]:
def rank_transform(X, ref=None):    """Transformasi rank Definisi 2.5. ref=fold latih untuk Protokol B."""    ref = X if ref is None else ref    out = np.empty(X.shape, dtype=float)    for k in range(X.shape[1]):        s = np.sort(ref[:, k])        out[:, k] = np.searchsorted(s, X[:, k], side='right') / len(s)    return outdef make_representations(F):    qt = QuantileTransformer(output_distribution='uniform',                             n_quantiles=min(1000, len(F)),                             random_state=0)    return {        'raw': F,        'minmax': MinMaxScaler().fit_transform(F),        'zscore': StandardScaler().fit_transform(F),        'rank': rank_transform(F),        'quantile': qt.fit_transform(F),    }

### Metrik dan protokol evaluasi

In [ ]:
def hungarian_map(y_true, cl, labs):    """Pemetaan cluster ke kelas yang optimal (Kuhn 1955). Noise -1 diabaikan."""    valid = cl >= 0    if valid.sum() == 0:        return np.array([labs[0]] * len(cl))    cids = np.unique(cl[valid])    cm = np.zeros((len(labs), len(cids)))    for i, l in enumerate(labs):        for j, c in enumerate(cids):            cm[i, j] = np.sum((y_true == l) & (cl == c))    r, c = linear_sum_assignment(-cm)    m = {cids[c[i]]: labs[r[i]] for i in range(len(r))}    return np.array([m.get(x, '__noise__') for x in cl])def external_metrics(y, yp, cl):    p, r, f, _ = precision_recall_fscore_support(        y, yp, average='weighted', zero_division=0)    return dict(acc=accuracy_score(y, yp) * 100, prec=p * 100, rec=r * 100,                f1=f * 100, ari=adjusted_rand_score(y, cl),                nmi=normalized_mutual_info_score(y, cl))def internal_metrics(X, cl):    if len(np.unique(cl[cl >= 0])) < 2:        return dict(sil=np.nan, db=np.nan, ch=np.nan)    m = cl >= 0    return dict(sil=silhouette_score(X[m], cl[m]),                db=davies_bouldin_score(X[m], cl[m]),                ch=calinski_harabasz_score(X[m], cl[m]))

In [ ]:
def protocol_A(X, y, labs, K, n_seeds=N_SEEDS, tag=''):    """Protokol A: 100 seed k-means++ pada seluruh data."""    rows = []    t0 = time.time()    for seed in range(n_seeds):        cl = KMeans(K, init='k-means++', n_init=1,                    random_state=seed).fit_predict(X)        rec = external_metrics(y, hungarian_map(y, cl, labs), cl)        if seed < N_SEEDS_INTERNAL:            rec.update(internal_metrics(X, cl))        rows.append(rec)    df = pd.DataFrame(rows)    out = {'representation': tag, 'n_seeds': n_seeds,           'runtime_s': round(time.time() - t0, 2)}    for c in df.columns:        v = df[c].dropna().values        out[f'{c}_mean'] = v.mean()        out[f'{c}_sd'] = v.std(ddof=1)    a = df['acc'].values    out['acc_min'], out['acc_max'] = a.min(), a.max()    out['acc_ci_lo'], out['acc_ci_hi'] = np.percentile(a, [2.5, 97.5])    return out, df['acc'].valuesdef protocol_B(F, y, labs, K, transform='raw', n_seeds=N_SEEDS_CV):    """Protokol B: 5-fold, transformasi di-fit HANYA pada fold latih."""    skf = StratifiedKFold(5, shuffle=True, random_state=0)    accs = []    for seed in range(n_seeds):        for tr, te in skf.split(F, y):            if transform == 'raw':                Xtr, Xte = F[tr], F[te]            elif transform == 'minmax':                sc = MinMaxScaler().fit(F[tr])                Xtr, Xte = sc.transform(F[tr]), sc.transform(F[te])            elif transform == 'zscore':                sc = StandardScaler().fit(F[tr])                Xtr, Xte = sc.transform(F[tr]), sc.transform(F[te])            elif transform == 'rank':                Xtr, Xte = rank_transform(F[tr]), rank_transform(F[te], ref=F[tr])            elif transform == 'quantile':                qt = QuantileTransformer(output_distribution='uniform',                                         n_quantiles=min(1000, len(tr)),                                         random_state=0).fit(F[tr])                Xtr, Xte = qt.transform(F[tr]), qt.transform(F[te])            km = KMeans(K, init='k-means++', n_init=1, random_state=seed).fit(Xtr)            cl = km.predict(Xte)            accs.append(accuracy_score(                y[te], hungarian_map(y[te], cl, labs)) * 100)    a = np.array(accs)    return dict(transform=transform, n_runs=len(a), acc_mean=a.mean(),                acc_sd=a.std(ddof=1),                acc_ci_lo=np.percentile(a, 2.5),                acc_ci_hi=np.percentile(a, 97.5))

### Grid search baseline clustering

In [ ]:
def grid_search_baselines(X, y, labs, K, n_seeds=30):    """Grid search dengan kriteria seleksi BEBAS LABEL (silhouette).    Memakai akurasi untuk memilih hyperparameter akan membocorkan label dan    membuat perbandingan tidak adil. Silhouette dipilih karena tersedia untuk    keempat algoritma. Konfigurasi terpilih lalu dievaluasi terhadap label.    """    results = []    # --- K-Means (metode yang diusulkan) ---    accs, sils, t0 = [], [], time.time()    for seed in range(n_seeds):        cl = KMeans(K, init='k-means++', n_init=1, random_state=seed).fit_predict(X)        accs.append(accuracy_score(y, hungarian_map(y, cl, labs)) * 100)        sils.append(silhouette_score(X, cl))    results.append(dict(method='K-Means (k-means++)', config=f'K={K}',                        acc_mean=np.mean(accs), acc_sd=np.std(accs, ddof=1),                        sil=np.mean(sils), runtime_s=(time.time() - t0) / n_seeds))    # --- Gaussian Mixture: grid pada covariance_type dan inisialisasi ---    # init_params bawaan sklearn adalah 'kmeans' dengan random_state yang sama,    # sehingga GMM konvergen ke partisi yang identik dengan K-Means dan    # perbandingannya menjadi tidak informatif. Kami menguji kedua inisialisasi    # dan melaporkan keduanya.    for init in ['kmeans', 'random_from_data']:        best = None        for cov in ['full', 'tied', 'diag', 'spherical']:            s = []            for seed in range(10):                cl = GaussianMixture(K, covariance_type=cov, init_params=init,                                     random_state=seed, n_init=1).fit_predict(X)                s.append(silhouette_score(X, cl))            if best is None or np.mean(s) > best[1]:                best = (cov, np.mean(s))        cov = best[0]        accs, sils, t0 = [], [], time.time()        for seed in range(n_seeds):            cl = GaussianMixture(K, covariance_type=cov, init_params=init,                                 random_state=seed, n_init=1).fit_predict(X)            accs.append(accuracy_score(y, hungarian_map(y, cl, labs)) * 100)            sils.append(silhouette_score(X, cl))        results.append(dict(            method=f'Gaussian mixture ({init})', config=f'cov={cov}',            acc_mean=np.mean(accs), acc_sd=np.std(accs, ddof=1),            sil=np.mean(sils), runtime_s=(time.time() - t0) / n_seeds))    # --- Agglomerative: grid pada linkage (deterministik, SD = 0) ---    best = None    for link in ['ward', 'complete', 'average']:        cl = AgglomerativeClustering(n_clusters=K, linkage=link).fit_predict(X)        s = silhouette_score(X, cl)        if best is None or s > best[1]:            best = (link, s, cl)    link, sil, cl = best    t0 = time.time()    AgglomerativeClustering(n_clusters=K, linkage=link).fit_predict(X)    results.append(dict(        method='Agglomerative', config=f'linkage={link}',        acc_mean=accuracy_score(y, hungarian_map(y, cl, labs)) * 100,        acc_sd=0.0, sil=sil, runtime_s=time.time() - t0))    # --- DBSCAN: grid pada eps dan min_samples (deterministik) ---    # Rentang eps disesuaikan skala fitur: koordinat ECDF berada di [0,1],    # sehingga grid 0.1-1.0 dari naskah lama terlalu lebar.    # Batasan penting: hanya konfigurasi yang menghasilkan tepat K klaster yang    # dipertimbangkan. Tanpa batasan ini seleksi berbasis silhouette memilih    # eps sangat kecil yang memecah data menjadi puluhan klaster berisi titik    # kembar, memberi silhouette = 1.0 yang degenerate. Deskriptor ECDF banyak    # mengandung nilai kembar sehingga masalah ini pasti muncul.    span = np.linalg.norm(X.max(axis=0) - X.min(axis=0))    best = None    for eps in np.linspace(0.01, 0.40, 40) * span:        for ms in [5, 10, 20, 30, 50]:            cl = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)            k_found = len(np.unique(cl[cl >= 0]))            if k_found != K or (cl < 0).mean() > 0.5:                continue            m = cl >= 0            s = silhouette_score(X[m], cl[m])            if best is None or s > best[2]:                best = (eps, ms, s, cl, k_found)    if best is None:        results.append(dict(method='DBSCAN', config='tidak ada konfigurasi valid',                            acc_mean=np.nan, acc_sd=np.nan, sil=np.nan,                            runtime_s=np.nan))    else:        eps, ms, s, cl, k_found = best        t0 = time.time()        DBSCAN(eps=eps, min_samples=ms).fit_predict(X)        results.append(dict(            method='DBSCAN',            config=f'eps={eps:.3f}, min_samples={ms}, K ditemukan={k_found}, '                   f'noise={(cl < 0).mean() * 100:.1f}%',            acc_mean=accuracy_score(y, hungarian_map(y, cl, labs)) * 100,            acc_sd=0.0, sil=s, runtime_s=time.time() - t0))    return pd.DataFrame(results)

## 3. Muat dataJika sel ini melempar `ValueError`, berkas Excel Anda menyimpan piksel sebagairepr numpy terpotong dan harus diekstrak ulang dengan`str(row['R_values'].tolist())`.

In [ ]:
imgs, y, filenames = load_pixels(EXCEL)labs = np.unique(y)K = len(labs)print(f"{len(imgs)} citra, ukuran {imgs.shape[1]}x{imgs.shape[2]}, {K} kelas")print(pd.Series(y).value_counts().to_dict())print("nama berkas unik:", len(set(filenames)) == len(filenames))

## 4. Baseline fiturMengisi TODO pertama: colour moments dan histogram RGB terkuantisasi.Sekaligus menjawab pertanyaan grid tujuh koordinat versus sembilan.

In [ ]:
descriptors = {    "ECDF-7 (diusulkan)":           feat_ecdf7(imgs),    "ECDF-9 (grid penuh)":          feat_ecdf9(imgs),    "Colour moments":               feat_colour_moments(imgs),    f"RGB histogram {HIST_BINS}^3": feat_rgb_histogram(imgs, HIST_BINS),}rows = []for name, F in descriptors.items():    X = StandardScaler().fit_transform(F) if F.shape[1] > 20 else F    r, _ = protocol_A(X, y, labs, K, n_seeds=N_SEEDS, tag=name)    r["d"] = F.shape[1]    rows.append(r)    print(f"{name:26s} d={F.shape[1]:5d}  "          f"acc={r['acc_mean']:.2f}+-{r['acc_sd']:.2f}  ARI={r['ari_mean']:.3f}")tab_desc = pd.DataFrame(rows)[    ["representation", "d", "acc_mean", "acc_sd", "acc_min", "acc_max",     "ari_mean", "nmi_mean", "sil_mean", "db_mean", "ch_mean"]]tab_desc.to_csv(f"{OUTDIR}/tabel_deskriptor.csv", index=False)tab_desc.round(3)

## 5. Perbandingan representasi dan uji t berpasangan

In [ ]:
F7 = descriptors["ECDF-7 (diusulkan)"]reps = make_representations(F7)rows, acc_by_rep = [], {}for name, X in reps.items():    r, a = protocol_A(X, y, labs, K, n_seeds=N_SEEDS, tag=name)    rows.append(r); acc_by_rep[name] = a    print(f"{name:10s} acc={r['acc_mean']:.2f}+-{r['acc_sd']:.2f}  "          f"ARI={r['ari_mean']:.3f}  sil={r['sil_mean']:.3f}  DB={r['db_mean']:.3f}")tab_A = pd.DataFrame(rows)[    ["representation", "acc_mean", "acc_sd", "acc_ci_lo", "acc_ci_hi",     "acc_max", "ari_mean", "nmi_mean", "sil_mean", "db_mean", "ch_mean"]]tab_A.to_csv(f"{OUTDIR}/tabel_protokol_A.csv", index=False)tab_A.round(3)

In [ ]:
tt = []for a in reps:    for b in reps:        if a >= b:            continue        d = acc_by_rep[a] - acc_by_rep[b]        t, p = stats.ttest_rel(acc_by_rep[a], acc_by_rep[b])        tt.append(dict(A=a, B=b, mean_diff_pp=d.mean(), t=t, p=p,                       cohen_dz=d.mean() / d.std(ddof=1)                                if d.std(ddof=1) > 0 else np.nan))tab_t = pd.DataFrame(tt)tab_t.to_csv(f"{OUTDIR}/tabel_uji_t.csv", index=False)tab_t.round(4)

## 6. Protokol BCross-validation lima lipatan, 20 seed. Transformasi di-fit hanya pada foldlatih, sehingga transformasi rank tidak membocorkan informasi dari fold uji.Sekitar tiga menit.

In [ ]:
rows = [protocol_B(F7, y, labs, K, transform=t)        for t in ["raw", "minmax", "zscore", "rank", "quantile"]]tab_B = pd.DataFrame(rows)tab_B.to_csv(f"{OUTDIR}/tabel_protokol_B.csv", index=False)tab_B.round(3)

## 7. Grid search baseline clusteringMengisi TODO ketiga. Hyperparameter dipilih dengan silhouette, bukan akurasi,agar label tidak bocor ke baseline.

In [ ]:
tab_gs = grid_search_baselines(F7, y, labs, K, n_seeds=30)tab_gs.to_csv(f"{OUTDIR}/tabel_grid_search.csv", index=False)tab_gs.round(3)

## 8. ResNet-50 (opsional)Perlu `torch`, `torchvision`, `Pillow`. Jika belum terpasang, jalankan selinstalasi di bawah sekali, lalu **restart kernel** dan ulangi dari sel 1.Dijalankan dua kali:* **8a** memakai citra asli dari `IMAGE_ROOT`. Ini perbandingan yang jujur  untuk ResNet dan yang akan diminta reviewer.* **8b** memakai citra 32x32 hasil rekonstruksi yang di-upsample. Ini  perbandingan input-identik dengan ECDF-7.Laporkan keduanya di naskah. Tanpa GPU, tiap tahap memakan 10--20 menit.

In [ ]:
# Hapus tanda pagar, jalankan sekali, lalu restart kernel:# %pip install torch torchvision pillow

In [ ]:
# --- 8a. citra asli ---F_resnet_orig = feat_resnet50(image_root=IMAGE_ROOT, filenames=filenames)print("dimensi fitur:", F_resnet_orig.shape[1])X = StandardScaler().fit_transform(F_resnet_orig)r_orig, _ = protocol_A(X, y, labs, K, n_seeds=N_SEEDS,                       tag="ResNet-50 (citra asli)")print(f"acc={r_orig['acc_mean']:.2f}+-{r_orig['acc_sd']:.2f}  "      f"ARI={r_orig['ari_mean']:.3f}")

In [ ]:
# --- 8b. citra 32x32 di-upsample (input identik dengan ECDF-7) ---F_resnet_32 = feat_resnet50(imgs=imgs)X = StandardScaler().fit_transform(F_resnet_32)r_32, _ = protocol_A(X, y, labs, K, n_seeds=N_SEEDS,                     tag="ResNet-50 (32x32 upsampled)")print(f"acc={r_32['acc_mean']:.2f}+-{r_32['acc_sd']:.2f}  "      f"ARI={r_32['ari_mean']:.3f}")tab_resnet = pd.DataFrame([r_orig, r_32])tab_resnet.to_csv(f"{OUTDIR}/tabel_resnet.csv", index=False)tab_resnet[["representation", "acc_mean", "acc_sd", "ari_mean",            "sil_mean", "runtime_s"]].round(3)

## 9. Ekspor ke LaTeXKeluaran sel ini bisa langsung ditempel ke `sn-article.tex`.

In [ ]:
def to_latex(df, caption, label, cols=None, digits=2):    d = df[cols] if cols else df    print(d.to_latex(index=False, float_format=f"%.{digits}f",                     caption=caption, label=label,                     column_format="@{}l" + "c" * (d.shape[1] - 1) + "@{}"))to_latex(tab_desc, "Perbandingan deskriptor, Protokol A.", "tab:descriptors",         ["representation", "d", "acc_mean", "acc_sd", "ari_mean", "sil_mean"])

In [ ]:
to_latex(tab_gs, "Baseline clustering dengan grid search.", "tab:clustering",         ["method", "config", "acc_mean", "acc_sd", "sil", "runtime_s"], digits=3)

In [ ]:
to_latex(tab_B, "Protokol B: cross-validation bebas kebocoran.", "tab:protocolB",         ["transform", "acc_mean", "acc_sd", "acc_ci_lo", "acc_ci_hi"])